# Using Tools with Claude

In [94]:
from dotenv import load_dotenv
import anthropic
from anthropic import Anthropic

print(f"Using Anthropic API: {anthropic.__version__}")

load_dotenv(override=True)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

Using Anthropic API: 1.1.0


## Modified Helper functions

We'll have to modify our `add_user_message()`, `add_assistant_message()` and `chat()` functions to handle multiple tool calls.

In [95]:
# modified helper functions

from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

### Define the tool functions

Let's define a tool function to get current date & time in a given format.

In [96]:
from datetime import datetime, timedelta
from anthropic.types import ToolParam


def add_duration_to_datetime(
    datetime_str: str, duration: timedelta, unit: str = "days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be null")
    return datetime.now().strftime(date_format)

In [97]:
add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

get_current_datetime_schema = ToolParam(
    {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string. This tool provides the current system time formatted as a string. Use this tool when you need to know the current date and time, such as for timestamping records, calculating time differences, or displaying the current time to users. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes. For example, '%Y-%m-%d' returns just the date in YYYY-MM-DD format, '%H:%M:%S' returns just the time in HH:MM:SS format, '%B %d, %Y' returns a date like 'May 07, 2025'. The default is '%Y-%m-%d %H:%M:%S' which returns a complete timestamp like '2025-05-07 14:32:15'.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }
)

In [98]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [ ]:
# some test calls
print(f"Default: {get_current_datetime()}")
print(f"Custom: {get_current_datetime('%d/%m/%Y %I:%M %p')}")

datetime_str = get_current_datetime()
print(datetime_str)
# print(add_duration_to_datetime(datetime_str, 5, "days"))
# print(add_duration_to_datetime(datetime_str, 1, "months"))

Default: 2026-09-08 14:14:45
Custom: 08/09/2026 02:14 PM
2026-09-08 14:14:45


ValueError: unconverted data remains:  14:14:45

In [ ]:
# def add_duration_to_datetime(
#     datetime_str: str, duration: timedelta, unit: str = "days", input_format="%Y-%m-%d"
a

We will also need to create a JSON schema describing the tool call function & params. This can be generated using Claude AI. Following schema was generated by Claude AI, which we assign to a varible, named with the same name as the tool function and ending with `_schema`.


An easy way to create the schema is to ask Claude to generate it. Head over to [claude.ai](https://claude.ai) and type in the following prompt and paste the tool function below the prompt:

`"Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation available at https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview.md"`

📌 **NOTE**: as of Aug 2026, the URL for Anthropic's tool documentation is - `https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview.md` - this could change in future. Paste in the correct URL.

The cell below shows me what I got back from Claude as the tool schema.

In [77]:
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

Now let's call Claude with this tool schema (JSON) and a user query. 

In [78]:
messages = []

messages.append(
    {"role": "user", "content": "What is the exact time formatted as HH:MM:SS?"}
)

response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=messages,
    tools=[get_current_datetime_schema],
)
messages.append({"role": "assistant", "content": response.content})

print(response)

Message(id='msg_011Cequ2wbZL6QNJuAzV8MH7', container=None, content=[ToolUseBlock(id='toolu_013yZJUFgDxtaVYGw2kR2Ej2', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=618, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


Here's how you'd run the tool itself. This code is written like this because Anthropic does not guarantee that every multi-block will always contain a text block.

In [79]:
from typing import NamedTuple, List, Dict
from anthropic.types import Message


class ToolCall(NamedTuple):
    id: str
    name: str
    input: dict


def get_tool_calls(response: Message) -> list[ToolCall]:
    """Return a ToolCall for every tool_use block in the response, or []."""
    if response.stop_reason != "tool_use":
        return []
    return [
        ToolCall(block.id, block.name, block.input)
        for block in response.content
        if block.type == "tool_use"
    ]


def make_tools_calls(response: Message) -> List[Dict]:
    """calls the respective tool and packs the result as expected by Claude"""

    tool_blocks = []
    if response.stop_reason == "tool_use":
        # tool_blocks = [block for block in response.content if block.type == "tool_use"]
        tool_blocks = get_tool_calls(response)

    # now call our tool - if tool_blocks is [], then this block will
    # not return anything!
    tool_call_results = []
    for tool_block in tool_blocks:
        name = tool_block.name  # which tool Claude wants to call
        args = tool_block.input  # dict of arguments for that tool
        tool_use_id = tool_block.id  # needed when you send the result back

        # check which tool function was asked for
        # in this example, there is just 1, but this could unwind to
        # a block like this...

        if name == "get_current_datetime":
            print(f"Calling tool {name} with args: {args}")
            result = get_current_datetime(**args)
            print(f"Result from {name}: {result}")
        # elsif name == "another_tool_name":
        #     result = another_tool_function(**args)

        # NOTE: only one of the tool function will be called at any time
        tool_call_result = {
            "type": "tool_result",
            "tool_use_id": tool_use_id,
            "content": result,
            "is_error": False,
        }
        tool_call_results.append(tool_call_result)

    return tool_call_results

In [80]:
# make the tool call
tool_call_results = make_tools_calls(response)
print(tool_call_results)

Calling tool get_current_datetime with args: {'date_format': '%H:%M:%S'}
Result from get_current_datetime: 11:31:50
[{'type': 'tool_result', 'tool_use_id': 'toolu_013yZJUFgDxtaVYGw2kR2Ej2', 'content': '11:31:50', 'is_error': False}]


Now we need to pass back the tool call result to Claude & get it's response. Here is how you do that.

In [81]:
messages.append({"role": "user", "content": tool_call_results})
print(messages)

[{'role': 'user', 'content': 'What is the exact time formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_013yZJUFgDxtaVYGw2kR2Ej2', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_013yZJUFgDxtaVYGw2kR2Ej2', 'content': '11:31:50', 'is_error': False}]}]


In [82]:
final_response = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(final_response.content[0].text)

The exact time is **11:31:50** (HH:MM:SS format).
